# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List record sets and their fields (All by `@id`)
record_sets = metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  name: {rs.get('name', '<no name>')}")
    print(f"  Fields:")
    for field in rs.get('fields', []):
        print(f"    Field @id: {field['@id']}")
        print(f"      label: {field.get('label', field.get('name', 'N/A'))}")
        print(f"      dataType: {field.get('dataType', 'N/A')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Build the list of record set @id's
record_set_ids = [rs['@id'] for rs in metadata.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Only build DataFrame if data returned
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for: {record_set_id} with columns: {df.columns.tolist()}")
    else:
        print(f"No records found for record set: {record_set_id}")
        dataframes[record_set_id] = pd.DataFrame()

# Example: show columns and first records for the main protein table
if dataframes:
    # Use the first non-empty dataframe
    for main_record_set_id, df in dataframes.items():
        if not df.empty:
            break
    print(f"\nColumns for record set {main_record_set_id}: \n{df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Identify a numeric field by @id from the schema. Let's guess a common one such as peptide count or molecular weight.
# Replace with actual @id from the printed list above.

# For demonstration, we'll use common field names. Adjust accordingly.
main_df = df  # from cell above

numeric_candidates = ['MW', 'peptide_count', 'coverage', 'unique_peptides', 'NormalizedAbundance']

# Detect which numeric field exists
import numpy as np
for field in numeric_candidates:
    if field in main_df.columns:
        numeric_field_id = field
        break
else:
    # fallback to first numeric
    num_cols = main_df.select_dtypes(include=[np.number]).columns
    if len(num_cols) > 0:
        numeric_field_id = num_cols[0]
    else:
        numeric_field_id = main_df.columns[0]  # fallback to first column

print(f"Using numeric field: {numeric_field_id}")

# Threshold for filtering
threshold = main_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 10
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (showing up to 5 rows):")
display(filtered_df.head())

# Z-score normalization
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping by a categorical field (e.g. description, accession, or group/sample field)
group_candidates = ['description', 'accession', 'SampleGroup', 'peptide']
for group_field in group_candidates:
    if group_field in main_df.columns:
        break
else:
    group_field = None

if group_field:
    print(f"\nGrouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in main_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=40, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field and group_field in main_df.columns:
        plt.figure(figsize=(10,5))
        # Display only the most common groups if many
        group_order = main_df[group_field].value_counts().index[:10]
        sns.boxplot(x=group_field, y=numeric_field_id, data=main_df[main_df[group_field].isin(group_order)])
        plt.title(f'{numeric_field_id} by {group_field} (Top 10 Groups)')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we've demonstrated how to load, inspect, and analyze the FAIR² dataset describing mass spectrometry analysis of human extracellular vesicles.\
* The data reveals variation in protein (or peptide) features across samples and groups. Data processing such as filtering, normalization, and grouping is straightforward using `mlcroissant` and standard Python libraries.\
* Distribution and group-wise comparisons can help identify proteins of interest or quality control aspects for further downstream bioinformatics work.\
* For deeper biological analysis, see the documentation and use field `@id`s as demonstrated to ensure reproducible and robust workflows.